# Data Exploration & Discovery

This notebook walks through the synthetic CRM dataset, identifying data quality issues
and documenting findings that inform the scoring model design.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Load raw data
accounts = pd.read_csv('../data/raw/accounts.csv')
leads = pd.read_csv('../data/raw/leads.csv')
contacts = pd.read_csv('../data/raw/contacts.csv')
campaign_members = pd.read_csv('../data/raw/campaign_members.csv')

print(f"Accounts: {len(accounts)}")
print(f"Leads: {len(leads)}")
print(f"Contacts: {len(contacts)}")
print(f"Campaign Members: {len(campaign_members)}")

## 1. Entity Population Overview

In [ ]:
# Lead status distribution
print("=== Lead Status Distribution ===")
print(leads['lead_status'].value_counts())
print(f"\nConverted leads: {leads['is_converted'].sum()} ({leads['is_converted'].mean()*100:.1f}%)")
print(f"Disqualified: {leads['is_disqualified'].sum()}")
print(f"Opted out: {leads['has_opted_out'].sum()}")
print(f"Bounced: {leads['email_bounced'].sum()}")

In [ ]:
# Contact status distribution
print("=== Contact Status Distribution ===")
print(contacts['contact_status'].value_counts())
print(f"\nWith lead origin: {contacts['has_lead_origin'].sum()} ({contacts['has_lead_origin'].mean()*100:.1f}%)")
print(f"No longer with company: {contacts['no_longer_with_company'].sum()}")
print(f"Opted out: {contacts['has_opted_out'].sum()}")

## 2. Data Quality Issues

In [ ]:
# DQ-1: Broken conversion links
converted = leads[leads['is_converted'] == True]
broken = converted[converted['converted_contact_id'].isna()]
print(f"DQ-1: Broken Conversion Links")
print(f"  Converted leads: {len(converted)}")
print(f"  Missing contact_id: {len(broken)} ({len(broken)/max(len(converted),1)*100:.1f}%)")

In [ ]:
# DQ-2: Email duplication
all_emails = pd.concat([leads[['email']], contacts[['email']]])
dup_emails = all_emails[all_emails.duplicated(subset='email', keep=False)]
print(f"DQ-2: Email Duplication")
print(f"  Total email records: {len(all_emails)}")
print(f"  Records with duplicate emails: {len(dup_emails)}")
print(f"  Unique duplicate emails: {dup_emails['email'].nunique()}")

In [ ]:
# DQ-4: ETL-dominated creation timestamps
lead_dates = leads['created_date'].value_counts()
print(f"DQ-4: ETL Timestamps")
print(f"  Lead creation dates with >20 records:")
print(lead_dates[lead_dates > 20])
print(f"\n  % of leads on bulk dates: {lead_dates[lead_dates > 20].sum() / len(leads) * 100:.1f}%")

In [ ]:
# DQ-6: Non-prospect contamination
non_prospect_leads = leads[leads['job_persona'] == 'Non-Prospect']
non_prospect_contacts = contacts[contacts['job_persona'] == 'Non-Prospect']
null_persona_leads = leads[leads['job_persona'].isna()]
print(f"DQ-6: Non-Prospect Contamination")
print(f"  Leads marked Non-Prospect: {len(non_prospect_leads)} ({len(non_prospect_leads)/len(leads)*100:.1f}%)")
print(f"  Contacts marked Non-Prospect: {len(non_prospect_contacts)}")
print(f"  Leads with NULL persona: {len(null_persona_leads)} ({len(null_persona_leads)/len(leads)*100:.1f}%)")

In [ ]:
# DQ-7: Data completeness
print(f"DQ-7: Data Completeness")
print(f"  Leads missing title: {leads['title'].isna().sum()} ({leads['title'].isna().mean()*100:.1f}%)")
print(f"  Leads missing persona: {leads['job_persona'].isna().sum()} ({leads['job_persona'].isna().mean()*100:.1f}%)")
print(f"  Contacts missing title: {contacts['title'].isna().sum()} ({contacts['title'].isna().mean()*100:.1f}%)")

In [ ]:
# DQ-8: Automation-inflated engagement
email_cms = campaign_members[campaign_members['campaign_type'] == 'Email']
auto_sends = email_cms[email_cms['member_status'] == 'Sent']
print(f"DQ-8: Automation Inflation")
print(f"  Total campaign members: {len(campaign_members)}")
print(f"  Email campaigns: {len(email_cms)} ({len(email_cms)/len(campaign_members)*100:.1f}%)")
print(f"  Auto sends (status=Sent): {len(auto_sends)} ({len(auto_sends)/len(campaign_members)*100:.1f}%)")
print(f"  Real engagements (is_responded=True): {campaign_members['is_responded'].sum()}")

## 3. Engagement Distribution Analysis

In [ ]:
# Engagement per entity
eng_per_entity = campaign_members.groupby(['entity_id', 'entity_type']).agg(
    total=('cm_id', 'count'),
    real=('is_responded', 'sum')
).reset_index()

print("Engagement per entity (all types):")
print(eng_per_entity['total'].describe())
print(f"\nReal engagement per entity:")
print(eng_per_entity['real'].describe())

In [ ]:
# Engagement recency
campaign_members['response_date'] = pd.to_datetime(campaign_members['response_date'])
from datetime import datetime
NOW = datetime(2025, 5, 15)
campaign_members['days_ago'] = (NOW - campaign_members['response_date']).dt.days

real_cm = campaign_members[campaign_members['is_responded'] == True]
print("Days since real engagement (distribution):")
print(real_cm['days_ago'].describe())
print(f"\n% within 30 days: {(real_cm['days_ago'] <= 30).mean()*100:.1f}%")
print(f"% within 90 days: {(real_cm['days_ago'] <= 90).mean()*100:.1f}%")

## 4. Account Analysis

In [ ]:
print("=== Account Characteristics ===")
print(f"Named accounts: {accounts['is_named_account'].sum()} ({accounts['is_named_account'].mean()*100:.1f}%)")
print(f"ICP qualified: {accounts['is_icp_qualified'].sum()} ({accounts['is_icp_qualified'].mean()*100:.1f}%)")
print(f"Do-not-contact: {accounts['do_not_contact'].sum()}")
print(f"\nIntent score distribution:")
print(accounts['intent_score'].describe())

## 5. Key Findings Summary

- **~20% of conversions have broken links** (DQ-1) — engagement is split across entities
- **~80% of lead creation dates are ETL artifacts** (DQ-4) — `created_date` is unreliable
- **~13% non-prospect contamination** (DQ-6) — competitors/vendors in the database
- **Email automation dominates campaign members** (DQ-8) — raw counts are misleading
- **Engagement recency is the strongest signal** — most real engagement happens in recent 90 days
- **Named accounts with high intent are underweighted** by the legacy MQL system